In [ ]:
!pip install transformer_lens accelerate

In [2]:
"""
Cross-model generalization test: port the validated Endogeneity +
Policy-Relevance discovery pipeline (docs/phase2_validation_results.md,
docs/knowledge_unit_reconciliation.md) to Qwen-2.5-3B -- a genuinely
different model (much larger, chat-tuned, different architecture
family) from GPT-2 small, matching the spirit of the non-LLM track's
Stage 8 ("does this generalize, or was it a GPT-2-specific trick").

Grounded in real, validated machinery throughout:
  - Same conditional Granger F-test + PCA-reduced controls + BH-FDR
    as phase2_open_discovery_standalone.py (architecture-agnostic,
    reads model.cfg.n_layers/n_heads, no hardcoded 12).
  - Same donor-activation patching for Policy-Relevance (NOT random
    noise, NOT an unembedding-difference steering vector -- both were
    tried elsewhere in this project and found weak/uninformative).
  - Same KnowledgeUnitView / EvidenceManager / QuestionGenerator
    architecture, unchanged.

Explicitly NOT included in this first pass (deliberate scope-narrowing
for a much bigger model, not an oversight):
  - Head-level candidates. Qwen-2.5-3B has ~36 layers; a full
    layer+head sweep at this scale is far more expensive than GPT-2's
    156-candidate pass. This first run is LAYER-LEVEL ONLY (~36
    candidates). Head-level discovery is a follow-up once layer-level
    results are in hand.
  - Chat-template-formatted prompts. Uses raw completion format
    ("{T} + {n} ="), matching Phase 1/2's validated GPT-2 format, NOT
    Qwen's instruction-tuned chat format. Whether raw completions even
    work sensibly on a chat-tuned model is exactly the first thing
    section 1 below checks -- if baseline accuracy/behavior looks
    degenerate, that's the first thing to revisit, not a reason to
    silently switch to chat formatting without checking.

STATUS: UNTESTED, written outside Colab. Expect debugging -- more than
usual, since neither the exact TransformerLens model string for
Qwen-2.5-3B nor its exact behavior under this prompt format has been
verified here. Read every printed diagnostic before trusting numbers
downstream of it.
"""

import numpy as np
import torch
from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Optional
from scipy import stats
from sklearn.linear_model import Ridge
from transformer_lens import HookedTransformer

# =============================================================================
# 0. Model loading
#
# UNVERIFIED: the exact model string TransformerLens expects for
# Qwen-2.5-3B. Try in this order if the first fails -- TransformerLens's
# supported-model list changes between versions, and Qwen naming
# conventions (base vs -Instruct) matter:
#   "Qwen/Qwen2.5-3B", "Qwen/Qwen2.5-3B-Instruct", "qwen2.5-3b"
# Check https://transformerlensorg.github.io (or
# transformer_lens.loading_from_pretrained.OFFICIAL_MODEL_NAMES in the
# installed version) if all of these fail.
# =============================================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B"  # [CHANGED after repeated OOM on 3B] --
# from_pretrained_no_processing still internally upcasts the full state
# dict to float32 during ProcessWeights.process_weights (unconditionally,
# not gated by the no-processing flags), needing ~12GB+ just for that
# step -- a hard wall on Colab's free-tier GPU (~14.56GB total, some
# already used by CUDA context/overhead) for a 3B model, not something
# fixable from this script alone. 1.5B needs proportionally less for
# the same fp32 upcast step. If this ALSO OOMs, drop to
# "Qwen/Qwen2.5-0.5B" next -- same architecture family, smaller still.

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)

print(f"[setup] loading {MODEL_NAME} in float16 (a 3B model will not fit "
      f"in fp32 on a typical Colab GPU)...")
# [FIXED after a second OOM/crash] from_pretrained_no_processing alone
# wasn't enough -- the underlying HuggingFace load still likely builds
# a full fp32 copy on CPU before casting to fp16 (~12GB+ for a 3B
# model), exceeding Colab free-tier RAM (~12-13GB) even before the
# model reaches the GPU. Fix: load the HF model directly with
# low_cpu_mem_usage=True (uses HF's meta-device loading trick to avoid
# ever materializing a full fp32 copy), then hand that already-fp16
# model to TransformerLens instead of letting it load a second copy.
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

print("[setup] loading HuggingFace model directly with low_cpu_mem_usage=True "
      "and device_map='auto' (requires `accelerate` -- pip install it if this "
      "errors with an ImportError)...")
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, low_cpu_mem_usage=True, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = HookedTransformer.from_pretrained_no_processing(
    MODEL_NAME, hf_model=hf_model, tokenizer=tokenizer, device=DEVICE, dtype=torch.float16)
model.eval()

del hf_model
gc.collect()
torch.cuda.empty_cache()

N_LAYERS = model.cfg.n_layers
N_HEADS = model.cfg.n_heads
D_MODEL = model.cfg.d_model
print(f"[setup] loaded. n_layers={N_LAYERS}, n_heads={N_HEADS}, d_model={D_MODEL} "
      f"(sanity check: GPT-2 small was 12/12/768 -- these should be much bigger)")

RNG = np.random.default_rng(0)
LAG_ORDER = 3
ALPHA = 0.05


# =============================================================================
# 1. FIRST CHECK: does the raw "{a} + {b} =" format even produce sensible
#    completions on a chat-tuned model? Run BEFORE anything else -- if
#    this looks degenerate, everything downstream needs rethinking, not
#    silent patching-over.
# =============================================================================

def baseline_accuracy_check(operand_max=9, max_pairs=40):
    rng = np.random.default_rng(1)
    pairs = [(a, b) for a in range(operand_max + 1) for b in range(operand_max + 1 - a)]
    rng.shuffle(pairs)
    pairs = pairs[:max_pairs]

    results = []
    for a, b in pairs:
        prompt = f"{a} + {b} = "  # [FIXED] trailing space -- Qwen tokenizes
        # the space after '=' as its own high-confidence token; without
        # an explicit trailing space, top-1 was that bare space token,
        # not a digit (see qwen_tokenizer_diagnostic.py findings)
        tokens = model.to_tokens(prompt)
        logits = model(tokens)
        pred_id = int(logits[0, -1, :].argmax().item())
        pred_str = model.to_string(torch.tensor([pred_id])).strip()
        correct = str(a + b)
        results.append({"a": a, "b": b, "prompt": prompt,
                         "predicted": pred_str, "correct": correct,
                         "is_correct": pred_str == correct})
    n_correct = sum(r["is_correct"] for r in results)
    print(f"\n[baseline check] {n_correct}/{len(results)} correct "
          f"({100*n_correct/len(results):.1f}%) on raw '{{a}} + {{b}} =' completions")
    print("First 10 examples:")
    for r in results[:10]:
        mark = "OK" if r["is_correct"] else "XX"
        print(f"  [{mark}] {r['prompt']!r} -> predicted {r['predicted']!r}, "
              f"correct {r['correct']!r}")
    if n_correct / len(results) < 0.3:
        print("\n[WARNING] Low accuracy. Before proceeding, manually inspect "
              "a few completions above -- if predictions look like garbage "
              "(not even plausible-looking numbers), the raw completion "
              "format may not work well on this chat-tuned model, and the "
              "running-total task design below needs reconsidering (e.g. "
              "chat-template formatting) before trusting anything past "
              "this point.")
    return results


baseline_results = baseline_accuracy_check()


# =============================================================================
# 2. PORTED, UNCHANGED (architecture-agnostic): conditional Granger F-test
#    with PCA-reduced controls, HypothesisEngine-equivalent batch test,
#    BH-FDR. Identical logic to phase2_open_discovery_standalone.py.
# =============================================================================

def build_lagged(matrix: np.ndarray, lag: int) -> np.ndarray:
    T = matrix.shape[0]
    cols = [matrix[lag - k: T - k] for k in range(1, lag + 1)]
    return np.concatenate(cols, axis=1)


def ols_rss(y: np.ndarray, X: np.ndarray) -> float:
    X_design = np.concatenate([np.ones((X.shape[0], 1)), X], axis=1)
    beta, _, _, _ = np.linalg.lstsq(X_design, y, rcond=None)
    resid = y - X_design @ beta
    return float(resid @ resid)


def benjamini_hochberg(p_values: list, alpha: float = ALPHA) -> list:
    m = len(p_values)
    order = np.argsort(p_values)
    sorted_p = np.array(p_values)[order]
    thresholds = (np.arange(1, m + 1) / m) * alpha
    passed = sorted_p <= thresholds
    cutoff_rank = np.max(np.where(passed)) if passed.any() else -1
    reject = np.zeros(m, dtype=bool)
    if cutoff_rank >= 0:
        reject[order[: cutoff_rank + 1]] = True
    return reject.tolist()


def conditional_granger_f_test_pca(target_col, other_matrix, action_onehot,
                                    lag=LAG_ORDER, k_pca=5):
    y = target_col[lag:]
    n = y.shape[0]
    self_lags = build_lagged(target_col.reshape(-1, 1), lag)

    if other_matrix.shape[1] > 0:
        k_eff = min(k_pca, other_matrix.shape[1])
        centered = other_matrix - other_matrix.mean(axis=0)
        _, _, Vt = np.linalg.svd(centered, full_matrices=False)
        components = centered @ Vt[:k_eff].T
        other_lags = build_lagged(components, lag)
    else:
        other_lags = np.zeros((n, 0))

    action_contemporaneous = action_onehot[lag:]
    action_past_lags = build_lagged(action_onehot, lag)
    action_history = np.concatenate([action_contemporaneous, action_past_lags], axis=1)

    restricted_X = np.concatenate([self_lags, other_lags], axis=1)
    unrestricted_X = np.concatenate([restricted_X, action_history], axis=1)
    k_u = unrestricted_X.shape[1] + 1
    q = k_u - (restricted_X.shape[1] + 1)
    df2 = n - k_u

    rss_r = ols_rss(y, restricted_X)
    rss_u = ols_rss(y, unrestricted_X)
    if df2 <= 0 or rss_u <= 0:
        return float("nan"), 1.0, df2
    f_stat = max(((rss_r - rss_u) / q) / (rss_u / df2), 0.0)
    p_value = stats.f.sf(f_stat, q, df2)
    return f_stat, p_value, df2


def test_endogeneity_batch(matrix, action_onehot, candidate_ids, lag=LAG_ORDER,
                            k_pca=5, alpha=ALPHA):
    p_values, df2s = [], []
    for i in range(matrix.shape[1]):
        target = matrix[:, i]
        others = np.delete(matrix, i, axis=1)
        _, p, df2 = conditional_granger_f_test_pca(target, others, action_onehot,
                                                     lag=lag, k_pca=k_pca)
        p_values.append(p)
        df2s.append(df2)
    rejected = benjamini_hochberg(p_values, alpha=alpha)
    print(f"[Stage 1] df2 across {len(candidate_ids)} tests: "
          f"min={min(df2s)}, mean={np.mean(df2s):.1f}")
    return {candidate_ids[i]: {"p_value": p_values[i], "significant": rejected[i], "df2": df2s[i]}
            for i in range(len(candidate_ids))}


# =============================================================================
# 3. PORTED, UNCHANGED: WorldModel, SelfGraph, KnowledgeUnitView,
#    EvidenceManager, QuestionGenerator.
# =============================================================================

class NodeStatus(Enum):
    CANDIDATE = "candidate"
    ACCEPTED = "accepted"
    REJECTED = "rejected"
    UNRESOLVED = "unresolved"


@dataclass
class Node:
    signal_id: str
    status: NodeStatus
    endogeneity_p_value: Optional[float] = None
    best_handle_id: Optional[str] = None
    flip_rate: Optional[float] = None
    entity_id: Optional[int] = None
    evidence: dict = field(default_factory=dict)


class WorldModel:
    def __init__(self):
        self.nodes: Dict[str, Node] = {}

    def add_candidate(self, signal_id: str, p_value: float):
        if signal_id not in self.nodes:
            self.nodes[signal_id] = Node(signal_id=signal_id, status=NodeStatus.CANDIDATE,
                                          endogeneity_p_value=p_value)

    def accept(self, signal_id: str, handle_id: str, flip_rate: float, evidence=None):
        node = self.nodes[signal_id]
        node.status = NodeStatus.ACCEPTED
        node.best_handle_id = handle_id
        node.flip_rate = flip_rate
        if evidence:
            node.evidence.update(evidence)

    def reject(self, signal_id: str, handle_id=None, flip_rate=None):
        node = self.nodes[signal_id]
        node.status = NodeStatus.REJECTED
        if handle_id:
            node.best_handle_id = handle_id
            node.flip_rate = flip_rate

    def self_model_summary(self) -> str:
        by_status: Dict[NodeStatus, List[str]] = {}
        for sid, node in self.nodes.items():
            by_status.setdefault(node.status, []).append(sid)
        lines = [f"  {s.value:<12}: {len(ids)} -- {ids}"
                 for s, ids in by_status.items() if ids]
        return "\n".join(lines) if lines else "  (empty)"


class NodeKind(Enum):
    OBSERVATION = "observation"
    MECHANISM = "mechanism"
    HANDLE = "handle"
    ATOMIC_SIGNAL = "atomic_signal"
    COMPOSITE_SIGNAL = "composite_signal"
    DECOY = "decoy"
    UNKNOWN = "unknown"


@dataclass
class SelfGraphNode:
    node_id: str
    kind: NodeKind = NodeKind.UNKNOWN
    kind_confidence: float = 0.0
    evidence: dict = field(default_factory=dict)
    semantic_label: str = "UNKNOWN"
    semantic_confidence: float = 0.0
    semantic_evidence: dict = field(default_factory=dict)


class KnowledgeUnitView:
    def __init__(self, kind_node: SelfGraphNode, world_model: WorldModel):
        self._kind_node = kind_node
        self._wm = world_model

    @property
    def node_id(self): return self._kind_node.node_id
    @property
    def kind(self): return self._kind_node.kind
    @property
    def kind_confidence(self): return self._kind_node.kind_confidence
    @property
    def status(self):
        node = self._wm.nodes.get(self.node_id)
        return node.status if node else None
    @property
    def flip_rate(self):
        node = self._wm.nodes.get(self.node_id)
        return node.flip_rate if node else None

    def __repr__(self):
        return (f"KnowledgeUnitView({self.node_id!r}, kind={self.kind.value}, "
                f"status={self.status.value if self.status else None}, "
                f"flip_rate={self.flip_rate})")


POLICY_RELEVANCE_FLIP_THRESHOLD = 0.3  # same provisional value as the GPT-2 work


class EvidenceManager:
    def __init__(self, world_model: WorldModel, self_graph_nodes: Dict[str, SelfGraphNode]):
        self.wm = world_model
        self.sg_nodes = self_graph_nodes

    def _ensure_node(self, node_id: str) -> SelfGraphNode:
        if node_id not in self.sg_nodes:
            self.sg_nodes[node_id] = SelfGraphNode(node_id=node_id)
        return self.sg_nodes[node_id]

    def record_endogeneity(self, node_id: str, p_value: float, significant: bool):
        node = self._ensure_node(node_id)
        node.evidence["endogeneity_p_value"] = p_value
        if significant:
            self.wm.add_candidate(node_id, p_value)

    def record_intervention(self, node_id: str, handle_id: str, flip_rate: float, evidence=None):
        node = self._ensure_node(node_id)
        if flip_rate >= POLICY_RELEVANCE_FLIP_THRESHOLD:
            self.wm.accept(node_id, handle_id, flip_rate, evidence)
            node.kind = NodeKind.MECHANISM
            node.kind_confidence = min(1.0, flip_rate)
        else:
            self.wm.reject(node_id, handle_id, flip_rate)
            node.kind = NodeKind.DECOY
            node.kind_confidence = 0.5


class QuestionGenerator:
    def next_question(self, kuv: KnowledgeUnitView) -> Optional[str]:
        if kuv.status is None:
            return "is_internal"
        if kuv.status == NodeStatus.CANDIDATE:
            return "is_controllable"
        return None


# =============================================================================
# 4. Running-total environment -- same design as Phase 2, unchanged except
#    reading N_LAYERS/N_HEADS dynamically (already architecture-agnostic
#    in the original; no changes needed here beyond reuse).
# =============================================================================

class RunningTotalEnvironment:
    N_STUCK_WARN = 8

    def __init__(self, model, operand_range=(1, 4), start_range=(1, 4), seed=0):
        self.model = model
        self.operand_range = operand_range
        self.start_range = start_range
        self.rng = np.random.default_rng(seed)
        self.T = None
        self.n_anomalies = 0
        self.n_steps_run = 0
        self._stuck_count = 0
        self._last_T = None
        self._warned_stuck = False

    def reset(self):
        self.T = int(self.rng.integers(self.start_range[0], self.start_range[1] + 1))
        self.n_anomalies = 0
        self.n_steps_run = 0
        self._stuck_count = 0
        self._last_T = None
        self._warned_stuck = False

    def step(self) -> dict:
        n = int(self.rng.integers(self.operand_range[0], self.operand_range[1] + 1))
        prompt = f"{self.T} + {n} = "  # [FIXED] trailing space, see note above
        tokens = self.model.to_tokens(prompt)
        logits = self.model(tokens)
        next_token_id = int(logits[0, -1, :].argmax().item())
        next_token_str = self.model.to_string(torch.tensor([next_token_id]))
        try:
            T_next = int(next_token_str.strip())
            valid = 0 <= T_next <= 9
        except ValueError:
            valid = False
            T_next = None
        if not valid:
            self.n_anomalies += 1
            T_next_for_continuation = (self.T + n) % 10
        else:
            T_next_for_continuation = T_next

        if self._last_T is not None and T_next_for_continuation == self._last_T:
            self._stuck_count += 1
        else:
            self._stuck_count = 0
        self._last_T = T_next_for_continuation
        if self._stuck_count >= self.N_STUCK_WARN and not self._warned_stuck:
            print(f"  !! WARNING: T stuck at {T_next_for_continuation} for "
                  f"{self._stuck_count + 1} consecutive steps")
            self._warned_stuck = True

        result = {"T": self.T, "n": n, "prompt": prompt,
                  "generated_token_str": next_token_str,
                  "T_next_valid": valid, "T_next": T_next_for_continuation}
        self.T = T_next_for_continuation
        self.n_steps_run += 1
        return result


def action_onehot_from_digit(digit: int, n_categories: int = 10) -> list:
    row = [0.0] * n_categories
    row[digit] = 1.0
    return row


# =============================================================================
# 5. Calibration (layer-level only, this first pass) + trajectory collection
# =============================================================================

def calibrated_addition_pairs(operand_max=9, seed=7):
    rng = np.random.default_rng(seed)
    candidates = [(a, b) for a in range(operand_max + 1) for b in range(operand_max + 1 - a)]
    valid = []
    for a, b in candidates:
        ans_ids = model.to_tokens(f"{a + b}", prepend_bos=False)  # [FIXED] no
        # leading space -- Qwen tokenizes bare digits (no leading space) as
        # single tokens, but " {digit}" (WITH leading space) as two tokens
        # (space + digit) -- the leading-space check found zero valid
        # pairs for exactly this reason
        if ans_ids.shape[-1] == 1:
            valid.append((a, b))
    rng.shuffle(valid)
    return valid


def calibrate_layer_directions(calibration_pairs):
    layer_acts = {l: [] for l in range(N_LAYERS)}
    sums = []
    for a, b in calibration_pairs:
        prompt = f"{a} + {b} = "  # [FIXED] trailing space, see note above
        tokens = model.to_tokens(prompt)
        _, cache = model.run_with_cache(tokens)
        for l in range(N_LAYERS):
            layer_acts[l].append(cache["resid_post", l][0, -1, :].float().cpu().numpy())
        sums.append(a + b)
    sums = np.array(sums)
    dirs = {}
    for l in range(N_LAYERS):
        X = np.stack(layer_acts[l])
        ridge = Ridge(alpha=10.0)
        ridge.fit(X, sums)
        w = ridge.coef_
        dirs[l] = w / (np.linalg.norm(w) + 1e-8)
    return dirs


def collect_trajectory(env, layer_dirs, n_steps):
    rows, actions = [], []
    env.reset()
    for _ in range(n_steps):
        step_result = env.step()
        tokens = model.to_tokens(step_result["prompt"])
        _, cache = model.run_with_cache(tokens)
        row = {}
        for l in range(N_LAYERS):
            act = cache["resid_post", l][0, -1, :].float().cpu().numpy()
            row[f"layer{l}"] = float(np.dot(act, layer_dirs[l]))
        rows.append(row)
        actions.append(action_onehot_from_digit(step_result["T_next"]))
        del cache
    candidate_ids = [f"layer{l}" for l in range(N_LAYERS)]
    matrix = np.array([[row[cid] for cid in candidate_ids] for row in rows])
    return matrix, np.array(actions), candidate_ids, env.n_anomalies


# =============================================================================
# 6. Policy-Relevance via REAL donor patching (not steering vectors)
# =============================================================================

def policy_relevance_flip_rate(layer, n_trials=20, position=-1, seed=1):
    rng = np.random.default_rng(seed)
    flips = 0
    n_used = 0
    for _ in range(n_trials):
        T = int(rng.integers(1, 5))
        n_clean = int(rng.integers(1, 5))
        n_donor = int(rng.integers(1, 5))
        tries = 0
        while n_donor == n_clean and tries < 10:
            n_donor = int(rng.integers(1, 5))
            tries += 1
        if n_donor == n_clean:
            continue
        clean_prompt = f"{T} + {n_clean} = "  # [FIXED] trailing space
        donor_prompt = f"{T} + {n_donor} = "  # [FIXED] trailing space
        clean_tokens = model.to_tokens(clean_prompt)
        donor_tokens = model.to_tokens(donor_prompt)
        if clean_tokens.shape[-1] != donor_tokens.shape[-1]:
            continue
        n_used += 1
        clean_logits = model(clean_tokens)
        original_action = int(clean_logits[0, -1, :].argmax().item())
        _, donor_cache = model.run_with_cache(donor_tokens)

        def patch_hook(activation, hook, donor_cache=donor_cache, position=position):
            activation[:, position, :] = donor_cache[hook.name][:, position, :]
            return activation

        patched_logits = model.run_with_hooks(
            clean_tokens, fwd_hooks=[(f"blocks.{layer}.hook_resid_post", patch_hook)])
        patched_action = int(patched_logits[0, -1, :].argmax().item())
        if patched_action != original_action:
            flips += 1
    return flips / n_used if n_used > 0 else float("nan")


# =============================================================================
# 7. Main
# =============================================================================

N_STEPS = 200
BUDGET = N_LAYERS  # test every layer that passes Stage 1 -- layer-only
                    # scope keeps this affordable even at Qwen's scale

print(f"\n[calibration] fitting {N_LAYERS} layer-level probe directions...")
calib_pairs = calibrated_addition_pairs()
layer_dirs = calibrate_layer_directions(calib_pairs)

env = RunningTotalEnvironment(model, seed=0)
print(f"\n[collection] gathering {N_STEPS}-step trajectory...")
matrix, action_onehot, candidate_ids, n_anomalies = collect_trajectory(env, layer_dirs, N_STEPS)
print(f"[collection] done. anomalies: {n_anomalies}/{N_STEPS}")

print("\n[Stage 1] batch Endogeneity screening...")
endog_results = test_endogeneity_batch(matrix, action_onehot, candidate_ids)
n_sig = sum(r["significant"] for r in endog_results.values())
print(f"[Stage 1] {n_sig}/{len(candidate_ids)} layers passed Endogeneity")

world_model = WorldModel()
self_graph_nodes: Dict[str, SelfGraphNode] = {}
evidence_mgr = EvidenceManager(world_model, self_graph_nodes)
question_gen = QuestionGenerator()

for cid, r in endog_results.items():
    evidence_mgr.record_endogeneity(cid, r["p_value"], r["significant"])

to_test = sorted([cid for cid, r in endog_results.items() if r["significant"]],
                  key=lambda cid: endog_results[cid]["p_value"])[:BUDGET]

print(f"\n[Stage 2] Policy-Relevance (donor patching) on {len(to_test)} layers...")
for cid in to_test:
    layer_num = int(cid[len("layer"):])
    fr = policy_relevance_flip_rate(layer_num)
    handle_id = f"donor_activation@{cid}"
    evidence_mgr.record_intervention(cid, handle_id, fr)
    kuv = KnowledgeUnitView(self_graph_nodes[cid], world_model)
    print(f"  {cid}: flip_rate={fr:.3f} -> {kuv}")

print("\n" + "=" * 60)
print("WorldModel summary (Qwen-2.5-3B, layer-level, first pass):")
print(world_model.self_model_summary())

accepted = [cid for cid, n in world_model.nodes.items() if n.status == NodeStatus.ACCEPTED]
print(f"\nAccepted layers: {accepted}")
print("\nsemantic_label left UNKNOWN for all candidates, same discipline "
      "as the GPT-2 track (docs/knowledge_unit_reconciliation.md section 4) "
      "-- structural finding only, no claim about WHAT these layers compute.")

np.savez("qwen_layer_discovery_results.npz",
         candidate_ids=np.array(candidate_ids),
         p_values=np.array([endog_results[c]["p_value"] for c in candidate_ids]),
         significant=np.array([endog_results[c]["significant"] for c in candidate_ids]),
         accepted=np.array(accepted))
print("\nSaved qwen_layer_discovery_results.npz")

ModuleNotFoundError: No module named 'transformer_lens'

In [1]:
"""
Qwen diagnostic: is 'always predict 1' an absolute lock, or a dominant
default hiding a weaker real signal underneath? Same diagnostic style
as GPT-2's Phase 1 baseline check (P(correct answer), rank(correct
answer) -- not just argmax match) -- distinguishes "the model has zero
information about the operands" from "the model has some information,
but a strong prior token wins the argmax regardless."

Run in the SAME Colab session (reuses `model`).
"""

import torch

results = []
for a in range(10):
    for b in range(1, 5):  # matches operand_range used in the running-total env
        prompt = f"{a} + {b} = "
        tokens = model.to_tokens(prompt)
        logits = model(tokens)
        pred_id = int(logits[0, -1, :].argmax().item())
        pred_str = model.to_string(torch.tensor([pred_id])).strip()
        correct_answer = str(a + b)

        correct_token_ids = model.to_tokens(correct_answer, prepend_bos=False)
        if correct_token_ids.shape[-1] != 1:
            continue  # skip if the correct answer isn't single-token for this pair
        correct_id = int(correct_token_ids[0, 0].item())

        probs = torch.softmax(logits[0, -1, :].float(), dim=-1)
        correct_prob = float(probs[correct_id].item())
        correct_rank = int((logits[0, -1, :] > logits[0, -1, correct_id]).sum().item()) + 1

        # also check: what's the probability mass on '1' specifically,
        # to quantify how dominant the default is
        one_token_ids = model.to_tokens("1", prepend_bos=False)
        one_id = int(one_token_ids[0, 0].item())
        prob_of_one = float(probs[one_id].item())

        results.append({
            "a": a, "b": b, "correct": a + b, "predicted": pred_str,
            "is_correct": pred_str == correct_answer,
            "correct_prob": correct_prob, "correct_rank": correct_rank,
            "prob_of_one": prob_of_one,
        })

n_correct = sum(r["is_correct"] for r in results)
print(f"=== Qwen2.5-1.5B: is 'always predict 1' absolute or a dominant default? ===")
print(f"{n_correct}/{len(results)} correct ({100*n_correct/len(results):.1f}%)\n")

print(f"{'a+b':>6} | {'correct':>7} | {'predicted':>9} | {'P(correct)':>10} | "
      f"{'rank(correct)':>13} | {'P(\"1\")':>8}")
print("-" * 68)
for r in results[:20]:
    print(f"{r['a']}+{r['b']:<4} | {r['correct']:>7} | {r['predicted']:>9} | "
          f"{r['correct_prob']:>10.4f} | {r['correct_rank']:>13} | {r['prob_of_one']:>8.4f}")

mean_correct_prob = sum(r["correct_prob"] for r in results) / len(results)
mean_correct_rank = sum(r["correct_rank"] for r in results) / len(results)
mean_prob_of_one = sum(r["prob_of_one"] for r in results) / len(results)
top3_count = sum(1 for r in results if r["correct_rank"] <= 3)

print(f"\nMean P(correct answer): {mean_correct_prob:.4f}")
print(f"Mean rank(correct answer): {mean_correct_rank:.2f}")
print(f"Mean P('1') specifically: {mean_prob_of_one:.4f}")
print(f"Pairs with correct answer in top-3: {top3_count}/{len(results)}")

print("\nInterpretation guide:")
print("  - If P('1') is huge (e.g. >0.5) and roughly CONSTANT across all "
      "(a,b) pairs, '1' is close to an absolute default regardless of "
      "input -- the model isn't really engaging with the operands for "
      "this prompt shape at all.")
print("  - If P(correct) or rank(correct) varies meaningfully across "
      "pairs (even while '1' usually still wins outright), there IS a "
      "real, weaker signal about the operands underneath the dominant "
      "default -- patching could still find something even though "
      "greedy accuracy is near zero, same as happened with GPT-2 "
      "(12.5% accuracy, but real patching signal found anyway).")
print("  - If both P(correct) and its rank are flat/uninformative "
      "across pairs too, that's a stronger sign this exact prompt "
      "format may just not engage arithmetic-relevant computation in "
      "this model at all, and the task design (not just the threshold) "
      "needs reconsidering -- e.g. a few-shot prefix, or checking "
      "whether the -Instruct chat-tuned variant with proper chat "
      "formatting behaves differently.")

NameError: name 'model' is not defined